In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict,Literal
from dotenv import load_dotenv
from pydantic import BaseModel,Field

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI(model="gpt-4o-mini")

In [4]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive","negative"] = Field(description="Sentiment of the review")
    

In [11]:
class DiagnosisSchema(BaseModel):
    issue_type:Literal["UX","Performance","Bug","Support","Other"] = Field(description="The category of issue mentioned in the review")
    tone:Literal["angry","frustrated","disappointed","calm"] = Field(description="The emotional tone expressed by the user")
    urgency:Literal["low","medium","high"] = Field(description="How urgent or critical the issue appears to be")


In [12]:
structure_model = model.with_structured_output(SentimentSchema)
structure_model2 = model.with_structured_output(DiagnosisSchema)


In [6]:
prompt = "What is the sentiment of the following review - The software too bad"
structure_model.invoke(prompt)

SentimentSchema(sentiment='negative')

In [7]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal["positive","negative"]
    diagnosis:dict
    response:str

In [14]:
def find_sentiment(state:ReviewState):
    prompt = f"For the following review find out the sentiment \n {state['review']}"
    sentiment = structure_model.invoke(prompt).sentiment

    return {"sentiment":sentiment}

def check_sentiment(state:ReviewState) -> Literal["positive_response","run_diagnosis"]:
    if state['sentiment'] == "positive":
        return "positive_response"
    else:
        return "run_diagnosis"
    
def positive_response(state:ReviewState):
    prompt = f"""
        write a warm thank you message in response to this review: \n\n "{state['review']}\"\n
        Also, kindly ask the user to leave feedback on our website.
"""
    
    response = model.invoke(prompt).content

    return {"response":response}

def run_diagnosis(state:ReviewState):
    prompt = f"""
    Diagnose this negative review:\n\n{state["review"]}\n" 
    "Return issue_type, tone, and urgency.
"""
    
    response = structure_model2.invoke(prompt)
    return {"diagnosis":response.model_dump()}

def negative_response(state: ReviewState):
    diagnosis = state["diagnosis"]

    prompt = f"""
    You are a support assistant.

    The user had a '{diagnosis["issue_type"]}' issue,
    sounded '{diagnosis["tone"]}',
    and marked urgency as '{diagnosis["urgency"]}'.

    Write an empathetic, helpful resolution message.
    """

    response = model.invoke(prompt).content

    return {"response": response}



In [16]:
graph = StateGraph(ReviewState)

graph.add_node("find_sentiment",find_sentiment)
graph.add_node("positive_response",positive_response)
graph.add_node("run_diagnosis",run_diagnosis)
graph.add_node("negative_response",negative_response)

graph.add_edge(START,"find_sentiment")
graph.add_conditional_edges("find_sentiment",check_sentiment)
graph.add_edge("positive_response",END)
graph.add_edge("run_diagnosis","negative_response")
graph.add_edge("negative_response",END)

workflow = graph.compile()


In [18]:
initial_state = {
    "review":"this app is not amazing , i have been use it since last 1 month and the ux is awesome, disgusting work designer team"
}

workflow.invoke(initial_state)

{'review': 'this app is not amazing , i have been use it since last 1 month and the ux is awesome, disgusting work designer team',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'UX',
  'tone': 'disappointed',
  'urgency': 'medium'},
 'response': "Subject: We're Here to Help with Your UX Issue\n\nHi [User's Name],\n\nThank you for reaching out and sharing your concerns with us. I’m really sorry to hear that you’re experiencing a UX issue, and I completely understand how disappointing that can be.\n\nYour feedback is very important to us, and we want to make sure we resolve this for you as smoothly as possible. Could you please provide a bit more information about what you’re experiencing? Any specific details about what’s not working as expected will help us assist you better.\n\nIn the meantime, I hope we can work together to get this sorted out and improve your experience. Thank you for your patience and understanding.\n\nLooking forward to your reply!\n\nBest regards,\n\n[Yo